# Project 2A — public reference demo

This is a demo notebook, not a student submission or a completed staff solution. Its signal functions and supporting declarations come only from the browser client's previously public project2a.py reference. The original function bodies are retained. add_silence factors out only that reference's existing trailing 100 ms of zeros; the host still supplies its separate leading 100 ms.

The browser selects required declarations from code-cell source and executes them locally in Pyodide. Markdown and stored outputs are ignored. Host UTF-8/preamble/length framing and output bounds live separately in host_codec.py. Optional visualization is unsupported in the browser audio path.


In [ ]:
import math
import numbers
import numpy as np

fs = 44_100
RATES = (50, 100, 200)
MAX_SEND_SECONDS = 15
MAX_CAPTURE_SECONDS = 20

class DecodeError(ValueError):
    """Could not locate a bounded message; safe to show the reason to the user."""

def _rate(rate):
    if isinstance(rate, (bool, np.bool_)) or not isinstance(rate, numbers.Real) or rate not in RATES:
        raise ValueError("Choose 50, 100 or 200 bits/s on both devices.")
    return int(rate)

def _samples(signal):
    # Inspect size before converting/copying a possibly large input.
    array = np.asarray(signal)
    if array.ndim != 1 or not 0 < array.size <= fs * MAX_CAPTURE_SECONDS:
        raise ValueError("Audio must be a nonempty, one-dimensional recording of at most 20 seconds.")
    if array.dtype.kind not in "fiu" or not np.all(np.isfinite(array)):
        raise ValueError("Audio samples must be finite real numbers.")
    return array.astype(np.float64, copy=False)

def _sample_count(bits, rate):
    return (bits * fs + rate - 1) // rate


In [ ]:
def generate_modulated_signal(message, rate, fc):
    """Continuous-phase sine × bits[floor(n * rate / fs)], with no symbol loop."""
    rate = _rate(rate)
    if isinstance(fc, bool) or not isinstance(fc, numbers.Real) or not math.isfinite(fc) or not 0 < fc < fs / 2:
        raise ValueError("Carrier must be finite and between 0 and the Nyquist frequency.")
    bits = np.asarray(message)
    if bits.ndim != 1 or not 0 < bits.size <= rate * MAX_SEND_SECONDS:
        raise ValueError("Supply a nonempty bit vector lasting at most 15 seconds.")
    if bits.dtype.kind not in "biuf" or not np.all((bits == 0) | (bits == 1)):
        raise ValueError("The message vector must contain only 0 and 1.")
    n = np.arange(_sample_count(bits.size, rate))
    symbols = np.floor(n * rate / fs).astype(np.int64)
    return bits[symbols] * np.sin(2 * np.pi * fc * n / fs)

def add_silence(signal):
    """Factor the public reference's existing trailing 100 ms of zeros."""
    return np.concatenate((signal, np.zeros(4_410)))


In [ ]:
def detect_start(signal, visual_test=False):
    """First absolute-value crossing of signal RMS (scalar multiplier: 1).

    visual_test retains the starter's optional visualize hook if supplied by a host.
    """
    array = _samples(signal)
    peak = float(np.max(np.abs(array)))
    normalized = array / peak if peak else array  # scale only; no amplitude floor
    threshold = float(np.sqrt(np.mean(normalized ** 2)))
    crossings = np.flatnonzero(np.abs(normalized) > threshold)
    if not crossings.size:
        raise DecodeError("No clear signal start found. Try again in a quieter room.")
    start = int(crossings[0])
    if visual_test and callable(globals().get("visualize")):
        visualize(array, threshold * peak, start, None)
    return start

def demodulate(signal, rate, length):
    """Use the first RMS crossing and half the power of ALL remaining samples."""
    rate = _rate(rate)
    if isinstance(length, (bool, np.bool_)) or not isinstance(length, numbers.Integral) or not 0 < length <= rate * MAX_SEND_SECONDS:
        raise ValueError("Expected bit length must be a positive bounded integer.")
    array = _samples(signal)
    start_index = detect_start(array)
    length = int(length)
    count = _sample_count(length, rate)
    if len(array) - start_index < count:
        raise DecodeError("Incomplete recording. Start Receive before Send and wait for the tone to finish.")
    bit_threshold = np.mean(array[start_index:] ** 2) / 2
    power = array[start_index:start_index + count] ** 2
    groups = np.floor(np.arange(count) * rate / fs).astype(np.int64)
    energies = np.bincount(groups, weights=power, minlength=length)
    counts = np.bincount(groups, minlength=length)
    return (energies / counts > bit_threshold).astype(np.uint8)
